# 03 – Tác động của Nhiễu đến Lợi thế Lượng tử

Notebook này khảo sát tác động của nhiễu lượng tử lên hiệu năng thuật toán Grover,
trả lời câu hỏi trung tâm:

> *"Khi nào nhiễu làm mất đi hoàn toàn lợi thế lượng tử so với tìm kiếm cổ điển?"*

Nội dung:
1. So sánh các loại nhiễu (Depolarizing, Bit-Flip, Phase-Flip, Amplitude Damping, Readout)
2. Noise sweep: xác suất thành công vs mức nhiễu
3. Ngưỡng mất lợi thế lượng tử (quantum advantage threshold)
4. So sánh preset noise models (Low / Medium / High)
5. Export toàn bộ kết quả

## 0. Import

In [2]:
import sys
sys.path.append('..')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.grover import (
    run_grover_simulation,
    calculate_success_probability,
    get_theoretical_success_probability,
    quantum_search_queries,
    classical_search_expected_queries,
)
from src.noise_model import (
    NoiseConfig,
    build_noise_model,
    build_noise_model_from_preset,
    sweep_noise_levels,
    compare_noise_types,
    describe_noise_model,
    NOISE_PRESETS,
)
from src.analysis import analyze_noise_impact, RESULTS_DIR

# Cấu hình chung
N_QUBITS = 3
TARGET   = 5
N_STATES = 2 ** N_QUBITS
N_SHOTS  = 2048
K_OPT    = max(1, round(math.pi / 4 * math.sqrt(N_STATES)))

print(f'Cấu hình: n={N_QUBITS}, N={N_STATES}, target={TARGET}, k_opt={K_OPT}')
print(f'Xác suất lý tưởng: {get_theoretical_success_probability(N_QUBITS, K_OPT):.4f}')
print(f'Random baseline  : {1/N_STATES:.4f}')

Cấu hình: n=3, N=8, target=5, k_opt=2
Xác suất lý tưởng: 0.9453
Random baseline  : 0.1250


## 1. Các loại nhiễu lượng tử

| Loại nhiễu | Toán tử Kraus | Mô tả vật lý |
|---|---|---|
| Depolarizing | $(1-p)\rho + p\frac{I}{2}$ | Nhiễu tổng quát, trung bình hóa nhiều nguồn |
| Bit-Flip | $(1-p)\rho + pX\rho X$ | Lật bit |0⟩ ↔ |1⟩ |
| Phase-Flip | $(1-p)\rho + pZ\rho Z$ | Lật pha |+⟩ ↔ |-⟩ |
| Amplitude Damping | $K_0\rho K_0^\dagger + K_1\rho K_1^\dagger$ | Relaxation $T_1$: |1⟩ → |0⟩ |
| Readout Error | - | Sai số khi đo lường |


In [3]:
# In mô tả các preset noise model
for name, config in NOISE_PRESETS.items():
    print(describe_noise_model(config))
    print()

──────────────────────────────────────────────────
  NOISE MODEL: IDEAL
──────────────────────────────────────────────────
  (Không có nhiễu - mô phỏng lý tưởng)
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  NOISE MODEL: LOW_NOISE
──────────────────────────────────────────────────
  Depolarizing (1-qubit) : 0.100%
  Depolarizing (2-qubit) : 0.500%
  Readout P(1|0)         : 0.500%
  Readout P(0|1)         : 0.500%
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  NOISE MODEL: MEDIUM_NOISE
──────────────────────────────────────────────────
  Depolarizing (1-qubit) : 0.500%
  Depolarizing (2-qubit) : 2.000%
  Readout P(1|0)         : 1.000%
  Readout P(0|1)         : 2.000%
──────────────────────────────────────────────────

──────────────────────────────────────────────────
  NOISE MODEL: HIGH_NOISE
──────────────────────────────────────────────────
  Depolarizing (1-qubit) 

## 2. So sánh tác động các loại nhiễu (p = 1%)

In [4]:
# Chạy phân tích tác động nhiễu đầy đủ (export ra results/)
noise_result = analyze_noise_impact(
    n_qubits=N_QUBITS,
    target_index=TARGET,
    n_shots=N_SHOTS,
    export=True
)

# In bảng kết quả so sánh
print('\nKết quả so sánh các loại nhiễu (p=1%):')
print(f'{"Loại nhiễu":<28} {"P(success)":>12} {"Suy giảm (%)":>14}')
print('─' * 56)
for noise_name, vals in noise_result['comparison'].items():
    deg_pct = vals['degradation_relative'] * 100
    print(f'{noise_name:<28} {vals["probability"]:>12.4f} {deg_pct:>13.1f}%')


[4] Noise impact analysis (3 qubits)...
    A) Comparing noise types at p=0.01...
    B) Depolarizing noise sweep...
  -> Saved: results\04a_noise_comparison_n3.png
  -> Saved: results\04b_noise_sweep_n3.png
  -> Saved: results\04_noise_sweep_n3.csv
  -> Saved: results\04_noise_comparison_n3.csv

Kết quả so sánh các loại nhiễu (p=1%):
Loại nhiễu                     P(success)   Suy giảm (%)
────────────────────────────────────────────────────────
Lý tưởng (Không nhiễu)             0.9502          -0.5%
Depolarizing                       0.3169          66.5%
Bit-Flip                           0.6641          29.8%
Phase-Flip                         0.6147          35.0%
Amplitude Damping                  0.7686          18.7%
Readout Error                      0.9097           3.8%


## 3. Noise Sweep: Ngưỡng mất lợi thế lượng tử

In [5]:
# Quét nhiều loại nhiễu trên cùng một trục để so sánh
noise_levels = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2]
noise_types  = ['depolarizing', 'bit_flip', 'phase_flip', 'amplitude_damping']
colors_map   = {
    'depolarizing':       '#2196F3',
    'bit_flip':           '#F44336',
    'phase_flip':         '#9C27B0',
    'amplitude_damping':  '#FF9800',
}
labels_map = {
    'depolarizing':       'Depolarizing',
    'bit_flip':           'Bit-Flip (X)',
    'phase_flip':         'Phase-Flip (Z)',
    'amplitude_damping':  'Amplitude Damping',
}

fig, ax = plt.subplots(figsize=(10, 5))
all_sweep_data = {}

for ntype in noise_types:
    sweep = sweep_noise_levels(
        N_QUBITS, TARGET, noise_levels,
        n_shots=N_SHOTS, noise_type=ntype
    )
    all_sweep_data[ntype] = sweep['success_probs']
    ax.plot(noise_levels, sweep['success_probs'],
            color=colors_map[ntype], marker='o', markersize=5,
            linewidth=2, label=labels_map[ntype])

# Các đường tham chiếu
ideal_p = get_theoretical_success_probability(N_QUBITS, K_OPT)
ax.axhline(y=ideal_p, color='green', linestyle='--',
           linewidth=1.5, alpha=0.8, label=f'Lý tưởng = {ideal_p:.3f}')
ax.axhline(y=1/N_STATES, color='gray', linestyle=':',
           linewidth=1.5, alpha=0.8, label=f'Random = {1/N_STATES:.3f}')

ax.set_xlabel('Mức nhiễu p')
ax.set_ylabel('Xác suất thành công')
ax.set_title(f'Tác động của các loại nhiễu đến Grover\n'
             f'n={N_QUBITS} qubit, N={N_STATES}, k_opt={K_OPT}')
ax.legend(loc='upper right')
ax.set_xlim(-0.005, 0.205)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_noise_type_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Lưu CSV
df_sweep = pd.DataFrame({'noise_level': noise_levels, **all_sweep_data})
df_sweep.to_csv(RESULTS_DIR / 'nb03_noise_sweep_all_types.csv', index=False, float_format='%.4f')
print('Lưu: results/nb03_noise_type_sweep.png')
print('Lưu: results/nb03_noise_sweep_all_types.csv')

Lưu: results/nb03_noise_type_sweep.png
Lưu: results/nb03_noise_sweep_all_types.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_3612\2059734965.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. So sánh Low / Medium / High noise preset

In [6]:
preset_results = {}
preset_names   = ['ideal', 'low_noise', 'medium_noise', 'high_noise']
ideal_prob     = get_theoretical_success_probability(N_QUBITS, K_OPT)

print(f'{"Preset":<15} {"P(success)":>12} {"vs Lý tưởng":>14} {"vs Random":>12}')
print('─' * 55)

for preset in preset_names:
    if preset == 'ideal':
        nm = None
    else:
        nm = build_noise_model_from_preset(preset)

    counts = run_grover_simulation(N_QUBITS, TARGET, K_OPT, N_SHOTS, noise_model=nm)
    prob   = calculate_success_probability(counts, TARGET, N_QUBITS)
    preset_results[preset] = prob

    vs_ideal  = (prob / ideal_prob - 1) * 100  # % thay đổi so với lý tưởng
    vs_random = prob / (1/N_STATES)             # bội số so với random
    print(f'{preset:<15} {prob:>12.4f} {vs_ideal:>+13.1f}% {vs_random:>10.1f}×')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
preset_colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
bars = ax.bar(preset_names, list(preset_results.values()),
              color=preset_colors, alpha=0.85, edgecolor='white')

for bar, prob in zip(bars, preset_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{prob:.3f}', ha='center', fontsize=9)

ax.axhline(y=ideal_prob, color='green', linestyle='--',
           linewidth=1.5, label=f'Lý tưởng = {ideal_prob:.3f}')
ax.axhline(y=1/N_STATES, color='gray', linestyle=':',
           linewidth=1.5, label=f'Random = {1/N_STATES:.3f}')

ax.set_ylabel('Xác suất thành công')
ax.set_title(f'So sánh Noise Presets - Grover n={N_QUBITS}')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_preset_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Lưu CSV
pd.DataFrame([
    {'preset': k, 'probability': v, 'ideal_prob': ideal_prob, 'random_baseline': 1/N_STATES}
    for k, v in preset_results.items()
]).to_csv(RESULTS_DIR / 'nb03_preset_comparison.csv', index=False, float_format='%.4f')
print('Lưu: results/nb03_preset_comparison.png')
print('Lưu: results/nb03_preset_comparison.csv')

Preset            P(success)    vs Lý tưởng    vs Random
───────────────────────────────────────────────────────
ideal                 0.9429          -0.3%        7.5×
low_noise             0.8384         -11.3%        6.7×
medium_noise          0.5200         -45.0%        4.2×
high_noise            0.1641         -82.6%        1.3×
Lưu: results/nb03_preset_comparison.png
Lưu: results/nb03_preset_comparison.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_3612\1230888703.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Ngưỡng mất lợi thế lượng tử

In [7]:
# Tìm ngưỡng chính xác hơn bằng cách quét dày hơn
fine_levels  = np.linspace(0, 0.3, 30).tolist()
random_base  = 1 / N_STATES
threshold_p  = noise_result.get('advantage_threshold')

sweep_fine = sweep_noise_levels(
    N_QUBITS, TARGET, fine_levels, n_shots=1024, noise_type='depolarizing'
)

# Tìm ngưỡng
threshold_idx = next(
    (i for i, p in enumerate(sweep_fine['success_probs']) if p <= random_base * 1.05),
    None
)
threshold_val = fine_levels[threshold_idx] if threshold_idx else None

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fine_levels, sweep_fine['success_probs'],
        color='#2196F3', linewidth=2, label='Grover (Depolarizing)')
ax.axhline(y=ideal_prob, color='green', linestyle='--',
           linewidth=1.3, label=f'Lý tưởng = {ideal_prob:.3f}')
ax.axhline(y=random_base, color='red', linestyle=':',
           linewidth=1.5, label=f'Random = {random_base:.3f}')

if threshold_val is not None:
    ax.axvline(x=threshold_val, color='orange', linestyle='-.',
               linewidth=2, label=f'Ngưỡng ≈ p={threshold_val:.3f}')
    ax.fill_betweenx([0, 1.05], 0, threshold_val,
                     alpha=0.07, color='blue', label='Vùng có lợi thế')

ax.set_xlabel('Mức nhiễu Depolarizing p')
ax.set_ylabel('Xác suất thành công')
ax.set_title(f'Ngưỡng mất lợi thế lượng tử\nn={N_QUBITS} qubit, Depolarizing noise')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_advantage_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKết luận:')
print(f'  Tại p ≈ {threshold_val:.3f} ({threshold_val*100:.1f}%), '
      f'Grover mất lợi thế so với tìm kiếm ngẫu nhiên.')
print('Lưu: results/nb03_advantage_threshold.png')


Kết luận:
  Tại p ≈ 0.031 (3.1%), Grover mất lợi thế so với tìm kiếm ngẫu nhiên.
Lưu: results/nb03_advantage_threshold.png


C:\Users\admin\AppData\Local\Temp\ipykernel_3612\3083710777.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Tổng kết

**Nhận xét:**
- **Depolarizing** là loại nhiễu gây suy giảm mạnh nhất (tác động toàn diện X+Y+Z).
- **Phase-Flip** tác động mạnh hơn **Bit-Flip** do thuật toán Grover nhạy cảm hơn với nhiễu pha (Diffusion operator dựa trên Hadamard).
- **Readout error** ảnh hưởng ít hơn vì chỉ xảy ra tại bước đo cuối.
- Lợi thế lượng tử bị mất hoàn toàn khi nhiễu depolarizing vượt ngưỡng ~10-20% (tùy số qubit).

**Ý nghĩa:** Đây là lý do tại sao NISQ devices hiện tại chưa thể khai thác đầy đủ lợi thế của Grover ở quy mô lớn — cần error correction để duy trì lợi thế.